In [1]:
import pandas as pd
from utils import min_max_normalize
import numpy as np

# Carregar dados e preparar variáveis

In [11]:
df_res = pd.read_excel(r"E:\repos\pessoal\redem-index\outputs\df_ipp_v2f.xlsx")

vars = [
    ("tempo_atuacao_dias_cum_ln_norm", r"$TempoAtuacao$", 1, "Carreira"),
    ("fid_gerais_cum_norm", r"$FidelidadeGerais$", 1, "Carreira"),
    ("relatorias_ln_cum_norm", r"$ln(Relatoria)$", 2, "Especializacao"),
    ("pos_comiss_pr_cum_norm", r"$PresidenciaComissao$", 3, "Especializacao"),
]


cols = [v[0] for v in vars]

df = df_res.fillna(0)

df[cols].describe()

# df_final = df_res.iloc[:, 0:8].join(df_res[cols])

,tempo_atuacao_dias_cum_ln_norm,fid_gerais_cum_norm,relatorias_ln_cum_norm,pos_comiss_pr_cum_norm
count,6250.000000,6250.000000,6250.000000,6250.000000
mean,0.802122,0.185326,0.232938,0.050912
std,0.099591,0.191437,0.189315,0.120063
min,0.000000,0.000000,0.000000,0.000000
25%,0.773878,0.000000,0.000000,0.000000
50%,0.773878,0.142857,0.240281,0.000000
75%,0.848837,0.285714,0.388007,0.000000
max,1.000000,1.000000,1.000000,1.000000


# Aplicando os pesos

In [13]:
dims = {v[-1] for v in vars}

# Compute each dimension's total score (sum over weighted variables for that dimension)
for d in dims:
    dim_vars = [v for v in vars if v[-1] == d]
    # Apply theoretical weights before summing
    weighted_vars = []
    for var, _, w, _ in dim_vars:
        weighted_var_name = f"{var}_w"
        df[weighted_var_name] = df[var] * w
        weighted_vars.append(weighted_var_name)
    df[f'dim_{d}'] = df[weighted_vars].sum(axis=1)
    print(f"Variables for dimension {d}: {dim_vars} (weighted: {weighted_vars})")

# Create final IPP as the average of both dimension columns
dim_cols = [f'dim_{d}' for d in dims]

# Normalize dim columns using minmax
for col in dim_cols:
    df[f"{col}_norm"] = min_max_normalize(df[col])


norm_dim_columns = [f"{col}_norm" for col in dim_cols]
df['ipp_final_v4'] = df[norm_dim_columns].mean(axis=1)

# Calculando dimensões e índice final

Variables for dimension Carreira: [('tempo_atuacao_dias_cum_ln_norm', '$TempoAtuacao$', 1, 'Carreira'), ('fid_gerais_cum_norm', '$FidelidadeGerais$', 1, 'Carreira')] (weighted: ['tempo_atuacao_dias_cum_ln_norm_w', 'fid_gerais_cum_norm_w'])
Variables for dimension Especializacao: [('relatorias_ln_cum_norm', '$ln(Relatoria)$', 2, 'Especializacao'), ('pos_comiss_pr_cum_norm', '$PresidenciaComissao$', 3, 'Especializacao')] (weighted: ['relatorias_ln_cum_norm_w', 'pos_comiss_pr_cum_norm_w'])


In [15]:
df.to_excel(r"E:\repos\pessoal\redem-index\outputs\df_ipp_v4.xlsx", index=False)